# Домашнее задание 5. Оптимизация с равенствами

**Выдано:** 21 ноября 2026. **Срок сдачи:** 5 декабря 2026. **Максимум:** 10 баллов (+2 бонус). **Время:** около 2,5 часа, бонус — ещё 20–30 минут.

Три задачи: эллипс (условия оптимальности), цилиндрическая банка (свой метод Ньютона–Лагранжа), калибровка робота по измерениям (обобщённый Гаусс–Ньютон). Вы пишете три функции — общую функцию ККТ `make_kkt` на CasADi, метод Ньютона–Лагранжа и обёртку Гаусса–Ньютона — и задаёте $f$ и $g$ двух задач символьно. Задаче 3 нужны `make_kkt` и `newton_lagrange` из задачи 2. Всё необходимое есть в лекциях 8–9.

**Как сдавать.** Заполните ячейки с `# TODO` и ответы `_Ваш ответ:_ TODO`. Выполните ноутбук целиком (Restart Kernel → Run All), сохраните результаты и сдайте как `hw5/homework.ipynb` в вашем закрытом `numopt-<логин>` организации `optimisation-misis-2026`. Инструкция: [CONTRIBUTING.md](https://github.com/IlyaChichkanov/Numerical-Optimization/blob/main/CONTRIBUTING.md). Решения пишите самостоятельно. Числа округляйте разумно, объясняйте по результатам.

**Материал:** [лекция 8](https://github.com/IlyaChichkanov/Numerical-Optimization/blob/main/lectures/lecture08/lecture08.md) — Лагранж, LICQ, редуцированный гессиан; [лекция 9](https://github.com/IlyaChichkanov/Numerical-Optimization/blob/main/lectures/lecture09/lecture09.md) — ККТ, Ньютон–Лагранж и обобщённый Гаусс–Ньютон.

| Задача | Баллы | Основной результат |
|---|---:|---|
| 1. Эллипс | 2 | Найти и классифицировать стационарные точки |
| 2. Банка | 4 | Функция ККТ, свой метод Ньютона–Лагранжа и роль стартового множителя |
| 3. Калибровка | 4 | Функция ККТ и обобщённый Гаусс–Ньютон: почему здесь он не уступает точному методу |
| Бонус | +2 | Робот задом наперёд: куда приходят два метода из перевёрнутого старта |

Нужны NumPy, Matplotlib, CasADi (лекция 7 §5) и, в бонусе, SciPy. Задачи, данные, проверка `check_kkt` и функции для рисунков уже даны.

Запуск: в Google Colab первая ячейка сама поставит CasADi; локально — `uv sync --group casadi` в корне клона репозитория курса, затем `uv run jupyter lab`.

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("casadi") is None:          # Colab: поставить CasADi один раз
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "casadi"], check=True)

import casadi as ca
import numpy as np
import matplotlib.pyplot as plt

BLUE, ORANGE, AQUA, RED, VIOLET, GRAY = "#2a78d6", "#eb6834", "#1baf7a", "#e34948", "#4a3aa7", "#8a8985"
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .2,
                     "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
                     "axes.prop_cycle": plt.cycler(color=[BLUE, ORANGE, AQUA, VIOLET])})

def plot_ellipse():
    t = np.linspace(0, 2*np.pi, 300)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(2*np.cos(t), np.sin(t), color=BLUE)
    ax.set(xlabel="x", ylabel="y", title="Допустимое множество: эллипс")
    ax.axis("equal")
    plt.tight_layout()
    return ax

def check_kkt(kkt, f, g, x, lam, h=1e-4, tol=1e-5):
    """Сверяет kkt(x, lam) = (grad_f, J, H, g) с конечными разностями f и g (лекция 7).

    H — гессиан функции Лагранжа L = f - lam^T g. Печатает расхождение каждого блока
    и останавливается на первом неверном."""
    x, lam = np.asarray(x, float), np.asarray(lam, float)
    n, m = len(x), len(lam)
    values = kkt(x, lam)
    assert not any(v is Ellipsis for v in values), "в функции остались заготовки ...: заполните grad_f, J, H, g"
    blocks = dict(zip(("grad_f", "J", "H", "g"), (np.asarray(v, float) for v in values)))
    need = {"grad_f": (n,), "J": (m, n), "H": (n, n), "g": (m,)}
    for name, shape in need.items():
        assert blocks[name].shape == shape, f"{name}: размер {blocks[name].shape}, нужен {shape}"
    E = h * np.eye(n)
    L = lambda y: f(y) - lam @ g(y)
    ref = {"grad_f": np.array([(f(x + e) - f(x - e)) / (2*h) for e in E]),
           "J": np.array([(g(x + e) - g(x - e)) / (2*h) for e in E]).T,
           "H": np.array([[(L(x + a + c) - L(x + a - c) - L(x - a + c) + L(x - a - c)) / (4*h*h) for c in E] for a in E]),
           "g": g(x)}
    for name in need:
        err = np.max(np.abs(blocks[name] - ref[name])) / (1 + np.max(np.abs(ref[name])))
        print(f"{name:6s} расхождение {err:.1e}")
        assert err < tol, f"{name} не совпадает с конечными разностями: проверьте формулы этого блока"
    print("все четыре блока верны")

# Задача 2: банка, x = (r, h)
def cylinder_f(x):
    r, h = x
    return r**2 + r*h

def cylinder_g(x):
    r, h = x
    return np.array([r**2*h - 2])           # одно ограничение — вектор из одного числа

def plot_histories(histories, labels, title):
    fig, ax = plt.subplots(figsize=(7, 3.5))
    for hist, label in zip(histories, labels):
        ax.semilogy([max(float(a[2]), 1e-16) for a in hist], "-o", ms=4, label=label)
    ax.set(xlabel="Номер итерации", ylabel="Норма невязки ККТ", title=title)
    ax.legend()
    plt.tight_layout()
    return ax

# Ориентиры в локальной системе робота. Координаты намеренно не симметричны.
POINTS = np.array([[0., 0.], [1., 0.], [0., 1.], [2., 1.], [-1., 1.], [1.5, -.5]])
TRUE_ANGLE = .65
TRUE_SHIFT = np.array([.9, -.35])

def make_measurements(sigma=.035):
    phi = TRUE_ANGLE
    R = np.array([[np.cos(phi), -np.sin(phi)], [np.sin(phi), np.cos(phi)]])
    rng = np.random.default_rng(2026)
    return (R @ POINTS.T).T + TRUE_SHIFT + rng.normal(0, sigma, POINTS.shape)

def design_block(p):
    """A_i @ (c,s,t_x,t_y) = R(c,s) p + t."""
    px, py = p
    return np.array([[px, -py, 1., 0.], [py, px, 0., 1.]])

A = np.vstack([design_block(p) for p in POINTS])   # 12x4: A @ z — все R(c,s) p_i + t подряд

# Задача 3: калибровка, z = (c, s, t_x, t_y)
MEASURED = make_measurements()          # измеренные q_i, массив 6x2, шум sigma = 0.035
b = MEASURED.ravel()                    # все координаты q_i подряд — вектор из 12 чисел

def calibration_f(z):
    return 0.5 * np.sum((A @ z - b)**2)

def calibration_g(z):
    return np.array([0.5 * (z[0]**2 + z[1]**2 - 1)])

def plot_calibration(measured, z):
    pred = (A @ z).reshape(-1, 2)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(measured[:, 0], measured[:, 1], marker="x", s=60, color=RED, label="измерения")
    ax.scatter(pred[:, 0], pred[:, 1], s=40, color=BLUE, label="модель")
    for q, m in zip(measured, pred):
        ax.plot([q[0], m[0]], [q[1], m[1]], color=GRAY, lw=.9)
    ax.set(xlabel="X", ylabel="Y", title="Калибровка: измерения и модель")
    ax.axis("equal")
    ax.legend()
    plt.tight_layout()
    return ax

print("Готово: матрица измерений A имеет форму", A.shape)

## Задача 1. Две стационарные точки (2 балла, около 20 минут)

Нужно минимизировать высоту точки на эллипсе:

$$
\min_{x,y} f(x,y)=y,\qquad g(x,y)=\tfrac12(x^2+4y^2-4)=0.
$$

Используйте соглашение курса $L=f-\lambda g$.

**(а) 1 балл.** Выпишите $\nabla f$, $\nabla g$, условия Лагранжа и найдите **все** стационарные точки вместе с множителями.

**(б) 1 балл.** Проверьте LICQ в обеих точках. Найдите касательный вектор $Z$ единичной длины и число $Z^\top\nabla^2_{xx}LZ$ в каждой точке. Какие точки являются строгим локальным минимумом и максимумом? Сверьтесь с графиком.

*Полный балл:* расчёт обеих точек, LICQ и знак редуцированного гессиана с выводом. *Половина:* правильные точки без проверки условий.

In [ ]:
plot_ellipse(); plt.show()

_Ваш ответ:_ TODO

## Задача 2. Банка и Ньютон–Лагранж (4 балла, около 60 минут)

Закрытая цилиндрическая банка объёма $2\pi$ имеет радиус $r>0$ и высоту $h>0$. Её площадь поверхности равна $2\pi(r^2+rh)$. Сократим постоянный множитель:

$$
\min_{r,h>0} f(r,h)=r^2+rh,\qquad g(r,h)=r^2h-2=0.
$$

В обвязке эта задача — функции `cylinder_f(x)` и `cylinder_g(x)`, где $x=(r,h)\in\mathbb R^2$.

**Важно.** Метод работает только с равенством $r^2h=2$. Неравенства $r>0$, $h>0$ он не соблюдает: промежуточные точки могут оказаться с $r\le0$ или $h\le0$. Неравенства — тема следующих лекций.

**(а) 2 балла — функция ККТ и метод.**

*Функция ККТ — через CasADi.* Производные руками не выводим: их построит CasADi, как в лекции 7 §5. Напишите общую функцию `make_kkt(f, g, x, lam)`. На входе — символ переменных `x` ($n$ чисел), символ множителей `lam` ($m$ чисел) и выражения CasADi `f` и `g`, построенные из `x`. На выходе — функция `kkt(x, lam)`, которая по числам возвращает четыре блока:

- `grad_f` — градиент $\nabla f$, вектор из $n$ чисел;
- `J` — якобиан ограничений, матрица $m\times n$; при одном ограничении это строка, а не столбец;
- `H` — гессиан функции Лагранжа $L=f-\lambda^\top g$, матрица $n\times n$: $H=\nabla^2 f-\lambda\nabla^2 g$, знак как в лекции 9;
- `g` — значение ограничений, вектор из $m$ чисел.

Подсказки:

- `ca.gradient(f, x)` — градиент скалярного выражения по `x` (лекция 7 §5);
- `ca.jacobian(g, x)` — якобиан, матрица $m\times n$; `ca.gradient(g, x)` дал бы столбец $n\times1$;
- `ca.hessian(L, x)` возвращает **пару** (гессиан, градиент), поэтому нужен `[0]`;
- `ca.dot(lam, g)` — скалярное произведение $\lambda^\top g$.

Перевод в NumPy уже написан в заготовке. Затем задайте банку символьно: `f` и `g` через `r` и `h`, обычной арифметикой (`r**2`, `r*h`). Последняя строка ячейки, `cylinder_kkt = make_kkt(f, g, x, lam)`, уже написана. Ячейка с `check_kkt` под ней сверит каждый блок с конечными разностями функций `cylinder_f` и `cylinder_g` (лекция 7) и назовёт неверный.

*Метод.* Напишите общую функцию `newton_lagrange(kkt, x0, lam0, tol=1e-10, maxit=50)` — для любого числа переменных $n$ и ограничений $m$: она понадобится и в задаче 3. На каждом шаге `kkt(x, lam)` возвращает $(\nabla f,J,H,g)$. Решите

$$
\begin{pmatrix}H&-J^\top\\J&0\end{pmatrix}
\begin{pmatrix}p\\\Delta\lambda\end{pmatrix}
=-\begin{pmatrix}\nabla f-J^\top\lambda\\g\end{pmatrix}
$$

и обновите $x,\lambda$. Верните **список** $(x_k,\lambda_k,\|F_k\|)$ с начальной точкой; при $\|F_k\|\leq\mathrm{tol}$ или $k=\mathrm{maxit}$ закончите. Обрабатывать вырожденную матрицу $K$ и делать линейный поиск не нужно.

**(б) 2 балла — опыт.** Найдите оптимальную банку аналитически: подставьте $h=2/r^2$ в $f$. Затем запустите метод дважды из $(r,h)=(1.5,\ 1.0)$:

- с $\lambda^{(0)}=0$;
- со стартовым множителем по МНК, как в лекции 9 §2.4: $\lambda^{(0)}$ — решение задачи $\min_\lambda\lVert\nabla f(x^{(0)})-J(x^{(0)})^\top\lambda\rVert$, одна строка `np.linalg.lstsq`.

Для каждого запуска выведите число шагов, ответ, первую точку $x^{(1)}$ и остались ли все точки с $r,h>0$. Нарисуйте `plot_histories` для двух запусков. Почему при $\lambda^{(0)}=0$ нужно намного больше шагов? Посмотрите, куда ведёт первый шаг.

*Полный балл:* (а) — обе проверки проходят; (б) — аналитический ответ, два запуска, график и объяснение через стартовый множитель. *Половина:* правильные числа без объяснения.

In [ ]:
def make_kkt(f, g, x, lam):
    # f, g — выражения CasADi от символа x; lam — символ множителей
    assert f is not ... and g is not ..., "сначала задайте выражения f и g в ячейке задачи"
    L = ...               # TODO: функция Лагранжа L = f - lam^T g (скаляр)
    grad_f = ...            # TODO: градиент f по x, n чисел
    J = ...                 # TODO: якобиан g по x, матрица m x n
    H = ...                 # TODO: гессиан L по x, матрица n x n
    assert not any(v is Ellipsis for v in (L, grad_f, J, H)), "заполните строки L, grad_f, J, H"
    assert not isinstance(H, tuple), "ca.hessian возвращает пару (гессиан, градиент): возьмите [0]"
    fun = ca.Function("kkt", [x, lam], [grad_f, J, H, g])      # символы -> четыре блока
    def kkt(x_val, lam_val):                                   # числа на входе, массивы NumPy на выходе
        gf, Jv, Hv, gv = fun(x_val, lam_val)
        return np.ravel(gf.full()), Jv.full(), Hv.full(), np.ravel(gv.full())
    return kkt

In [ ]:
x = ca.SX.sym("x", 2)           # символ точки x = (r, h)
lam = ca.SX.sym("lam", 1)       # символ множителя: одно ограничение
r, h = x[0], x[1]
f = ...                         # TODO: f(r, h) через r и h
g = ...                         # TODO: g(r, h) через r и h
cylinder_kkt = make_kkt(f, g, x, lam)

In [ ]:
# Проверка cylinder_kkt: каждый блок против конечных разностей cylinder_f и cylinder_g в точке (1.3, 0.7) при lambda = 0.4
check_kkt(cylinder_kkt, cylinder_f, cylinder_g, (1.3, 0.7), [0.4])

In [ ]:
def newton_lagrange(kkt, x0, lam0, tol=1e-10, maxit=50):
    # TODO: ваш код
    raise NotImplementedError

In [ ]:
# Самопроверка общего метода: аналитический ответ банки — (r,h,lambda)=(1,2,1).
test = newton_lagrange(cylinder_kkt, [1.1, 1.7], [0.])
assert np.allclose(test[-1][0], [1., 2.], atol=1e-8), "Проверьте знаки в матрице ККТ"
assert np.allclose(test[-1][1], [1.], atol=1e-8) and test[-1][2] < 1e-8
print("Ньютон–Лагранж: проверка пройдена")

In [ ]:
# TODO: ваш код
# Два запуска из (1.5, 1.0), вывод чисел и график plot_histories(...)


_Ваш ответ:_ TODO

## Задача 3. Калибровка поворота и переноса (4 балла, около 60 минут)

У робота известны координаты шести ориентиров $p_i\in\mathbb R^2$ в его собственной системе координат. Камера измерила их глобальные положения $q_i$ **с шумом**. Нужно найти поворот и перенос, которые лучше всего объясняют измерения.

Параметризуем поворот числами $c,s$, а перенос — $(t_x,t_y)$. Переменные $z=(c,s,t_x,t_y)\in\mathbb R^4$:

$$
R(c,s)=\begin{pmatrix}c&-s\\s&c\end{pmatrix},\quad
g(z)=\tfrac12(c^2+s^2-1)=0,
$$

$$
\min_z f(z)=\frac12\sum_{i=1}^6\|R(c,s)p_i+t-q_i\|^2.
$$

Данные уже в обвязке: `MEASURED` — измеренные $q_i$ (шум $\sigma=0.035$), `b` — все их координаты подряд, вектор из 12 чисел. Матрица `A` размера $12\times4$ собрана так, что $Az$ — все $R(c,s)p_i+t$ подряд. Поэтому вектор невязок $r(z)=Az-b$, а его якобиан $J_r=A$. Функции `calibration_f(z)` и `calibration_g(z)` задают $f$ и $g$.

**(а) 1 балл — функция ККТ и Гаусс–Ньютон.**

*Функция ККТ.* Задайте $f$ и $g$ символьно и получите `calibration_kkt = make_kkt(f, g, z, lam)` — та же `make_kkt`, что в задаче 2, теперь для $z\in\mathbb R^4$. Подсказка: `A @ z - b` работает и с символом `z` — это вектор невязок $r(z)$; сумма квадратов — `ca.sumsqr`. Ячейка с `check_kkt` проверит блоки по функциям `calibration_f` и `calibration_g`.

*Гаусс–Ньютон.* Напишите `gauss_newton_kkt(kkt, jac_r)`: обёртку, которая возвращает те же $\nabla f$, $J$, $g$, но матрицу $H_{\mathrm{GN}}=J_r^\top J_r$ вместо точного гессиана. Цикл метода не переписывайте: `newton_lagrange(gauss_newton_kkt(kkt, jac_r), z0, lam0)` из задачи 2 и есть обобщённый Гаусс–Ньютон. Обёртка учебная: она вычисляет точный $H$ и подменяет его. В настоящих солверах (например, acados) вторые производные при этом не считают вовсе — в этом и экономия.

Если `make_kkt` или `newton_lagrange` из задачи 2 не работает, `gauss_newton_kkt` всё равно проверяется отдельной ячейкой ниже. За одну ошибку дважды не снижаем.

**(б) 3 балла — опыт.** Запустите оба метода из $z_0=(1,0,0,0)$, $\lambda_0=0$. Выведите число итераций, угол $\varphi=\operatorname{atan2}(s,c)$ и $\lambda^\ast$. Нарисуйте `plot_histories` для двух методов и `plot_calibration(MEASURED, z_star)` для найденного преобразования. Ответьте на два вопроса:

- что Гаусс–Ньютон выбрасывает из точной матрицы $H$;
- почему вблизи найденного решения можно ожидать похожего поведения Ньютона–Лагранжа и обобщённого Гаусса–Ньютона? Сравните $\lambda^\ast$ с собственными числами $A^\top A$ (`np.linalg.eigvalsh(A.T @ A)`).

*Полный балл:* (а) — обе проверки проходят; (б) — числа, два графика и объяснение через $\lambda^\ast$ и собственные числа $A^\top A$. *Половина:* числа без объяснения.

In [ ]:
z = ca.SX.sym("z", 4)           # символ z = (c, s, t_x, t_y)
lam = ca.SX.sym("lam", 1)       # символ множителя: одно ограничение
c, s = z[0], z[1]               # поворот; перенос z[2], z[3] в ограничение не входит
f = ...                         # TODO: f(z) = 1/2 * сумма квадратов невязок A @ z - b
g = ...                         # TODO: g(z) через c и s
calibration_kkt = make_kkt(f, g, z, lam)

In [ ]:
# Проверка calibration_kkt: каждый блок против конечных разностей calibration_f и calibration_g
check_kkt(calibration_kkt, calibration_f, calibration_g, (0.8, 0.6, 0.1, -0.2), [0.3])

In [ ]:
def gauss_newton_kkt(kkt, jac_r):
    # TODO: ваш код
    raise NotImplementedError

In [ ]:
# Проверка обёртки: изменяется только гессиан, остальные блоки остаются теми же.
x_test = np.array([.8, .6, .1, -.2])
lam_test = np.array([.3])
gf, J, H, g = calibration_kkt(x_test, lam_test)
gf2, J2, H2, g2 = gauss_newton_kkt(calibration_kkt, lambda z: A)(x_test, lam_test)
assert np.allclose(H2, A.T@A) and np.allclose(gf, gf2) and np.allclose(J, J2) and np.allclose(g, g2)
assert np.linalg.norm(H-H2)>1e-3, "Проверка должна отличать точный и приближённый гессианы"
print("Гаусс–Ньютон: проверка пройдена")

In [ ]:
# TODO: ваш код
# Оба метода из (1, 0, 0, 0); числа; plot_histories(...) и plot_calibration(MEASURED, z_star)


_Ваш ответ:_ TODO

## Бонус. Робот задом наперёд (до +2 баллов, 20–30 минут)

Калибровку часто начинают с грубой оценки ориентации. Пусть она ошиблась на $180^\circ$: старт $z_0=(-1,0,0,0)$, $\lambda_0=0$, данные те же, что в задаче 3.

**(а) +1 балл.** Запустите из этого старта оба метода и сохраните истории в `flipped_newton` (Ньютон–Лагранж) и `flipped_gauss_newton` (Гаусс–Ньютон). Для каждого ответа выведите угол $\varphi$ в градусах, $\lVert r\rVert$, $\lambda^\ast$ и наименьшее собственное число $Z^\top HZ$. Здесь $Z$ — `scipy.linalg.null_space(J)`, $H$ — точный гессиан из `calibration_kkt`. Нарисуйте `plot_calibration` для ответа Ньютона–Лагранжа. Что получилось?

**(б) +1 балл.** Почему из этого старта методы пришли к разным стационарным точкам? Означает ли это, что обобщённый Гаусс–Ньютон гарантированно находит минимум? Обоснуйте. Подсказка: запустите Гаусс–Ньютон из точки, куда пришёл Ньютон–Лагранж, — `newton_lagrange(gauss_newton_kkt(calibration_kkt, lambda z: A), z_nl, lam_nl)`.

Бонус необязателен. Если не решаете, оставьте ячейки как есть.

In [ ]:
# бонус (необязательно): ваш код


_Бонус (необязательно):_ ваш ответ